<a href="https://colab.research.google.com/github/kanishka102008/DAA-Lab/blob/main/exp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import random
import string


def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    matches, comparisons = [], 0

    if m == 0 or m > n:
        return matches, comparisons

    for i in range(n - m + 1):
        j = 0

        while j < m:
            comparisons += 1

            if text[i + j] != pattern[j]:
                break

            j += 1

        if j == m:
            matches.append(i)

    return matches, comparisons


def compute_lps(pattern):
    m = len(pattern)
    lps = [0] * m

    length = 0
    i = 1

    while i < m:

        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1

        elif length != 0:
            length = lps[length - 1]

        else:
            lps[i] = 0
            i += 1

    return lps


def kmp_search(text, pattern):
    n, m = len(text), len(pattern)

    if m == 0 or m > n:
        return [], 0

    lps = compute_lps(pattern)

    matches = []
    comparisons = 0

    i = 0
    j = 0

    while i < n:

        comparisons += 1

        if pattern[j] == text[i]:
            i += 1
            j += 1

            if j == m:
                matches.append(i - j)
                j = lps[j - 1]

        elif pattern[j] != text[i]:

            if j != 0:
                j = lps[j - 1]

            else:
                i += 1

    return matches, comparisons


def rabin_karp(text, pattern, q=101):
    n, m = len(text), len(pattern)

    if m == 0 or m > n:
        return [], 0

    d = 256

    h = pow(d, m - 1, q)

    p_hash = 0
    t_hash = 0

    matches = []
    comparisons = 0

    # Calculate initial hashes
    for i in range(m):
        p_hash = (d * p_hash + ord(pattern[i])) % q
        t_hash = (d * t_hash + ord(text[i])) % q

    # Search
    for s in range(n - m + 1):

        if p_hash == t_hash:

            for k in range(m):

                comparisons += 1

                if text[s + k] != pattern[k]:
                    break

            else:
                matches.append(s)

        # Calculate next hash
        if s < n - m:

            t_hash = (
                d * (t_hash - ord(text[s]) * h)
                + ord(text[s + m])
            ) % q

            if t_hash < 0:
                t_hash += q

    return matches, comparisons


# --------------------------------
# Main Execution
# --------------------------------

text = 'AABAACAADAABAABA'
pattern = 'AABA'

print("Text:", text)
print("Pattern:", pattern)

m1, c1 = naive_search(text, pattern)
m2, c2 = kmp_search(text, pattern)
m3, c3 = rabin_karp(text, pattern)

print()
print("Naive -> Matches at:", m1, ", Comparisons:", c1)
print("KMP -> Matches at:", m2, ", Comparisons:", c2)
print("RK -> Matches at:", m3, ", Comparisons:", c3)


# --------------------------------
# Performance Comparison
# --------------------------------

text_large = ''.join(
    random.choices('ABCD', k=10000)
)

patterns = [
    'AB',
    'ABCD',
    'ABCDAB',
    'ABCDABCD'
]

print()
print(
    f'{"Pattern":>12} '
    f'{"Naive":>10} '
    f'{"KMP":>10} '
    f'{"RK":>10}'
)

print('-' * 50)

for p in patterns:

    _, c1 = naive_search(text_large, p)
    _, c2 = kmp_search(text_large, p)
    _, c3 = rabin_karp(text_large, p)

    print(
        f'{p:>12} '
        f'{c1:>10} '
        f'{c2:>10} '
        f'{c3:>10}'
    )

Text: AABAACAADAABAABA
Pattern: AABA

Naive -> Matches at: [0, 9, 12] , Comparisons: 30
KMP -> Matches at: [0, 9, 12] , Comparisons: 20
RK -> Matches at: [0, 9, 12] , Comparisons: 12

     Pattern      Naive        KMP         RK
--------------------------------------------------
          AB      12500      11908       1186
        ABCD      13236      12467        219
      ABCDAB      13278      12499        150
    ABCDABCD      13279      12501        136
